# 02 — DuckDB Spatial Join (replaces PostGIS)
Run with `uv run jupyter lab` or `uv run marimo run notebooks/02_duckdb_spatial_join.py` — zero server.

In [ ]:
import polars as pl, duckdb
duckdb.sql("INSTALL spatial; LOAD spatial")
duckdb.sql("SELECT count(*) FROM read_parquet('data/curated/exposures.parquet')").show()

In [ ]:
duckdb.sql("""
  CREATE OR REPLACE TABLE loss AS
  SELECT e.id, e.sector, e.value_cr, e.EAD_cr, h.depth_m,
         CASE e.sector WHEN 'Real Estate' THEN 0.35 WHEN 'Power' THEN 0.28 ELSE 0.22 END * h.depth_m * e.value_cr AS loss_cr,
         e.geometry
  FROM read_parquet('data/curated/exposures.parquet') e
  JOIN read_parquet('data/curated/hazard_flood_100yr.parquet') h ON ST_Intersects(e.geometry, h.geometry)
""")
duckdb.sql("COPY (SELECT * FROM loss) TO 'data/curated/loss.parquet' (FORMAT PARQUET)")
duckdb.sql("SELECT sector, count(*), round(sum(loss_cr),1) FROM loss GROUP BY sector ORDER BY 3 DESC").show()

In [ ]:
import leafmap.maplibregl as leafmap
m = leafmap.Map(center=[19.1,72.9], zoom=9, style="positron")
m.add_vector("data/curated/hazard_flood_100yr.geojson", layer_type="fill", fill_color="blue", fill_opacity=0.2)
m